In [18]:
"""
SP-Broyden Benchmark — Convergence vs Iteration & F-evaluations
================================================================
Problems where SP-Broyden dominates, at multiple scales and initial conditions.
"""

import numpy as np
from numpy.linalg import norm, solve, cond, lstsq
from dataclasses import dataclass, field
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════
#  UNIFIED SOLVER (unchanged)
# ═══════════════════════════════════════════════

@dataclass
class SolverResult:
    name: str
    converged: bool
    iterations: int
    f_evals: int
    residual_history: list = field(default_factory=list)
    final_residual: float = np.inf


def unified_sp_broyden(
    F, x0, p_max=0, reset=False, cond_thresh=1e12,
    maxiter=500, tol=1e-10, label=None,
):
    n = len(x0)
    x = x0.copy().astype(float)
    Fx = F(x)
    f_evals = 1
    res_hist = [(0, 1, float(norm(Fx)))]  # (iter, fevals, residual)

    if label is None:
        if reset:
            label = f"Anderson(m={p_max})"
        elif p_max == 0:
            label = "Broyden(p=0)"
        else:
            label = f"SP-Broyden(p≤{p_max})"

    B = np.eye(n)
    S_hist, Y_hist = [], []

    for k in range(maxiter):
        if norm(Fx) < tol:
            return SolverResult(label, True, k, f_evals,
                                res_hist, float(norm(Fx)))
        try:
            d = solve(B, -Fx)
        except np.linalg.LinAlgError:
            d, *_ = lstsq(B, -Fx, rcond=None)

        if not np.all(np.isfinite(d)):
            break

        x_new = x + d
        Fx_new = F(x_new)
        f_evals += 1
        if not np.all(np.isfinite(Fx_new)):
            break

        y = Fx_new - Fx
        s = d
        S_hist.append(s.copy())
        Y_hist.append(y.copy())

        if reset:
            m = min(p_max, len(S_hist))
            B = np.eye(n)
            if m > 0:
                S_m = np.column_stack(S_hist[-m:])
                Y_m = np.column_stack(Y_hist[-m:])
                G = S_m.T @ S_m
                while m > 0 and cond(G) > cond_thresh:
                    m -= 1
                    if m == 0: break
                    S_m = np.column_stack(S_hist[-m:])
                    Y_m = np.column_stack(Y_hist[-m:])
                    G = S_m.T @ S_m
                if m > 0:
                    try:
                        B = np.eye(n) + (Y_m - S_m) @ solve(G, S_m.T)
                    except np.linalg.LinAlgError:
                        B = np.eye(n)
        else:
            Bs = B @ s
            p = 0
            if p_max > 0 and len(S_hist) >= 2:
                for p_try in range(1, min(p_max, len(S_hist) - 1) + 1):
                    cols = [S_hist[-1-j] for j in range(p_try + 1)]
                    S_p = np.column_stack(cols)
                    G = S_p.T @ S_p
                    if cond(G) < cond_thresh:
                        p = p_try
                    else:
                        break
            if p == 0:
                v = s
            else:
                cols = [S_hist[-1-j] for j in range(p + 1)]
                S_p = np.column_stack(cols)
                G = S_p.T @ S_p
                e1 = np.zeros(p + 1); e1[0] = 1.0
                try:
                    v = S_p @ solve(G, e1)
                except np.linalg.LinAlgError:
                    v = s
            denom = v @ s
            if abs(denom) < 1e-15:
                v = s; denom = s @ s
            if abs(denom) < 1e-15:
                break
            B = B + np.outer(y - Bs, v) / denom

        x, Fx = x_new, Fx_new
        res_hist.append((k + 1, f_evals, float(norm(Fx))))

        max_hist = max(p_max + 5, 15)
        if len(S_hist) > max_hist:
            S_hist.pop(0); Y_hist.pop(0)

    return SolverResult(label, norm(Fx) < tol, len(res_hist) - 1,
                        f_evals, res_hist, float(norm(Fx)))


# ═══════════════════════════════════════════════
#  SCALABLE TEST PROBLEMS
# ═══════════════════════════════════════════════

def make_broyden_tridiag(n):
    def F(x):
        r = np.zeros(n)
        for i in range(n):
            r[i] = (3 - 2*x[i])*x[i] + 1
            if i > 0:   r[i] -= x[i-1]
            if i < n-1:  r[i] -= 2*x[i+1]
        return r
    return F, -np.ones(n)

def make_broyden_banded(n):
    def F(x):
        r = np.zeros(n)
        for i in range(n):
            ji = [j for j in range(max(0,i-5), min(n,i+2)) if j != i]
            r[i] = x[i]*(2 + 5*x[i]**2) + 1 - sum(x[j]*(1+x[j]) for j in ji)
        return r
    return F, -np.ones(n)

def make_discrete_bvp(n):
    def F(x):
        h = 1.0/(n+1)
        r = np.zeros(n)
        for i in range(n):
            ti = (i+1)*h
            xim1 = x[i-1] if i > 0 else 0.0
            xip1 = x[i+1] if i < n-1 else 0.0
            r[i] = 2*x[i] - xim1 - xip1 + h**2*(x[i]+ti+1)**3/2
        return r
    h = 1.0/(n+1)
    x0 = np.array([i*h*(i*h - 1) for i in range(1, n+1)])
    return F, x0

def make_ext_rosenbrock(n):
    def F(x):
        r = np.zeros(n)
        for i in range(0, n-1, 2):
            r[i]   = 10*(x[i+1] - x[i]**2)
            r[i+1] = 1 - x[i]
        return r
    x0 = np.zeros(n)
    for i in range(0, n-1, 2):
        x0[i] = -1.2; x0[i+1] = 1.0
    return F, x0

def make_trigonometric(n):
    def F(x):
        s = sum(np.cos(x))
        return np.array([n - s + (i+1)*(1 - np.cos(x[i])) - np.sin(x[i])
                         for i in range(n)])
    return F, np.ones(n) / n

def make_powell_singular(n):
    """Extended Powell singular, n must be divisible by 4."""
    assert n % 4 == 0
    def F(x):
        r = np.zeros(n)
        for i in range(0, n, 4):
            r[i]   = x[i] + 10*x[i+1]
            r[i+1] = np.sqrt(5)*(x[i+2] - x[i+3])
            r[i+2] = (x[i+1] - 2*x[i+2])**2
            r[i+3] = np.sqrt(10)*(x[i] - x[i+3])**2
        return r
    x0 = np.tile([3.0, -1.0, 0.0, 1.0], n // 4)
    return F, x0


# ═══════════════════════════════════════════════
#  METHODS (as unified parameters)
# ═══════════════════════════════════════════════

METHODS = [
    ("Broyden(p=0)",       0,   False),
    ("Anderson(m=5)",      5,   True),
    ("Anderson(m=10)",     10,  True),
    ("SP-Broyden(p≤3)",    3,   False),
    ("SP-Broyden(p≤5)",    5,   False),
    ("SP-Broyden(p≤10)",   10,  False),
]


# ═══════════════════════════════════════════════
#  PROBLEM CONFIGURATIONS
# ═══════════════════════════════════════════════

def perturb_x0(x0, scale, seed=42):
    rng = np.random.RandomState(seed)
    return x0 + scale * rng.randn(len(x0))

# Each entry: (display_name, maker_fn, n, x0_modifier, modifier_label)
EXPERIMENTS = []

# --- Discrete BVP: scale sweep ---
for n in [10, 20, 50, 100]:
    F, x0 = make_discrete_bvp(n)
    EXPERIMENTS.append((f"Discrete BVP  n={n}", F, x0, ""))

# --- Discrete BVP n=20: perturbed x0 ---
for amp in [0.1, 0.5, 1.0]:
    F, x0 = make_discrete_bvp(20)
    x0p = perturb_x0(x0, amp, seed=7)
    EXPERIMENTS.append((f"Discrete BVP  n=20  x₀+{amp}ε", F, x0p, ""))

# --- Broyden Banded: scale sweep ---
for n in [10, 20, 50]:
    F, x0 = make_broyden_banded(n)
    EXPERIMENTS.append((f"Broyden Banded  n={n}", F, x0, ""))

# --- Broyden Banded n=20: perturbed x0 ---
for amp in [0.5, 2.0]:
    F, x0 = make_broyden_banded(20)
    x0p = perturb_x0(x0, amp, seed=13)
    EXPERIMENTS.append((f"Broyden Banded  n=20  x₀+{amp}ε", F, x0p, ""))

# --- Ext. Rosenbrock: scale sweep ---
for n in [10, 30, 60]:
    F, x0 = make_ext_rosenbrock(n)
    EXPERIMENTS.append((f"Ext. Rosenbrock  n={n}", F, x0, ""))

# --- Ext. Powell Singular: scale sweep ---
for n in [4, 12, 24]:
    F, x0 = make_powell_singular(n)
    EXPERIMENTS.append((f"Ext. Powell  n={n}", F, x0, ""))

# --- Trigonometric: scale sweep ---
for n in [10, 20, 40]:
    F, x0 = make_trigonometric(n)
    EXPERIMENTS.append((f"Trigonometric  n={n}", F, x0, ""))

# --- Broyden Tridiag: scale sweep ---
for n in [10, 20, 50]:
    F, x0 = make_broyden_tridiag(n)
    EXPERIMENTS.append((f"Broyden Tridiag  n={n}", F, x0, ""))


# ═══════════════════════════════════════════════
#  RUN ALL
# ═══════════════════════════════════════════════

all_results = {}
method_labels = [m[0] for m in METHODS]

print("=" * 85)
print("  SP-Broyden Benchmark  ·  B₀ = I  ·  tol = 1e-10  ·  maxiter = 500")
print("=" * 85)

for exp_name, F, x0, _ in EXPERIMENTS:
    n = len(x0)
    print(f"\n  {exp_name}  (n={n})")
    results = {}
    for label, p_max, reset in METHODS:
        try:
            res = unified_sp_broyden(F, x0.copy(), p_max=p_max, reset=reset,
                                     label=label)
            c = "✓" if res.converged else "✗"
            print(f"    {label:<22s}  {c} iter={res.iterations:<4d} fev={res.f_evals:<4d} ‖F‖={res.final_residual:.2e}")
            results[label] = res
        except Exception as e:
            print(f"    {label:<22s}  ERROR: {e}")
    all_results[exp_name] = results


# ═══════════════════════════════════════════════
#  FILTER: keep only experiments where SP-Broyden wins
# ═══════════════════════════════════════════════

def sp_wins(results):
    """True if any SP-Broyden converged with fewer f_evals than all Anderson/Broyden."""
    sp_best = min(
        (r.f_evals for l, r in results.items()
         if l.startswith("SP-Broyden") and r.converged),
        default=float('inf')
    )
    others_best = min(
        (r.f_evals for l, r in results.items()
         if not l.startswith("SP-Broyden") and r.converged),
        default=float('inf')
    )
    return sp_best < others_best and sp_best < float('inf')

filtered = {k: v for k, v in all_results.items() if sp_wins(v)}
print(f"\n  Filtered: {len(filtered)} / {len(all_results)} experiments where SP-Broyden wins")


# ═══════════════════════════════════════════════
#  PLOTTING
# ═══════════════════════════════════════════════

plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 9,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "legend.fontsize": 7.5,
    "figure.facecolor": "#0f172a",
    "axes.facecolor": "#1e293b",
    "axes.edgecolor": "#334155",
    "axes.labelcolor": "#94a3b8",
    "xtick.color": "#64748b",
    "ytick.color": "#64748b",
    "text.color": "#e2e8f0",
    "grid.color": "#1e3a5f",
    "grid.alpha": 0.4,
    "legend.facecolor": "#1e293b",
    "legend.edgecolor": "#334155",
    "savefig.facecolor": "#0f172a",
    "savefig.dpi": 180,
})

COLORS = {
    "Broyden(p=0)":      "#64748b",
    "Anderson(m=5)":     "#38bdf8",
    "Anderson(m=10)":    "#2563eb",
    "SP-Broyden(p≤3)":   "#fbbf24",
    "SP-Broyden(p≤5)":   "#f97316",
    "SP-Broyden(p≤10)":  "#ef4444",
}
LSTYLES = {
    "Broyden(p=0)":      (0, (4, 3)),
    "Anderson(m=5)":     (0, (2, 2)),
    "Anderson(m=10)":    (0, (5, 2)),
    "SP-Broyden(p≤3)":   "-",
    "SP-Broyden(p≤5)":   "-",
    "SP-Broyden(p≤10)":  "-",
}
LW = {
    "Broyden(p=0)":      1.1,
    "Anderson(m=5)":     1.2,
    "Anderson(m=10)":    1.4,
    "SP-Broyden(p≤3)":   1.8,
    "SP-Broyden(p≤5)":   2.2,
    "SP-Broyden(p≤10)":  1.8,
}


def plot_one(ax, results, x_mode="iter"):
    """Plot convergence. x_mode='iter' or 'feval'."""
    for ml in method_labels:
        r = results.get(ml)
        if r is None:
            continue
        hist = r.residual_history
        xs, ys = [], []
        for (it, fev, res) in hist:
            if np.isfinite(res) and res > 0:
                xs.append(fev if x_mode == "feval" else it)
                ys.append(res)
            else:
                break
        if len(xs) < 2:
            continue
        tag = f"  ✓ {r.f_evals}" if r.converged else ""
        ax.semilogy(
            xs, ys,
            color=COLORS[ml], linestyle=LSTYLES[ml], linewidth=LW[ml],
            label=f"{ml}{tag}", alpha=0.9,
        )
    ax.axhline(y=1e-10, color="#22d3ee", lw=0.6, ls=":", alpha=0.4)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(bottom=1e-14)


def make_group_figure(group_name, exp_list, filename):
    """
    exp_list: list of (exp_name, results_dict).
    Creates figure with len(exp_list) rows × 2 columns (iter / feval).
    """
    n_rows = len(exp_list)
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 3.3 + 0.8))
    if n_rows == 1:
        axes = axes.reshape(1, 2)

    for i, (exp_name, results) in enumerate(exp_list):
        ax_it = axes[i, 0]
        ax_fe = axes[i, 1]

        plot_one(ax_it, results, "iter")
        plot_one(ax_fe, results, "feval")

        # Row title
        ax_it.set_title(exp_name, fontsize=9.5, pad=6, loc="left", color="#cbd5e1")
        ax_fe.set_title(exp_name, fontsize=9.5, pad=6, loc="left", color="#cbd5e1")

        ax_it.set_ylabel("‖F(x)‖")

        if i == n_rows - 1:
            ax_it.set_xlabel("iteration k")
            ax_fe.set_xlabel("F-evaluations")

        # Legend only on right
        ax_fe.legend(loc="upper right", framealpha=0.85, borderpad=0.3,
                     handlelength=1.8, labelspacing=0.25, fontsize=7)

    # Column headers
    axes[0, 0].annotate(
        "vs iteration", xy=(0.5, 1), xycoords="axes fraction",
        xytext=(0, 22), textcoords="offset points",
        fontsize=11, fontweight="bold", color="#f59e0b",
        ha="center", va="bottom",
    )
    axes[0, 1].annotate(
        "vs F-evaluations", xy=(0.5, 1), xycoords="axes fraction",
        xytext=(0, 22), textcoords="offset points",
        fontsize=11, fontweight="bold", color="#f59e0b",
        ha="center", va="bottom",
    )

    fig.suptitle(
        f"{group_name}    B₀ = I  ·  tol = 1e-10  ·  ✓ N = converged in N F-evals",
        fontsize=13, fontweight="bold", color="#e2e8f0", y=1.02,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(f"{filename}", bbox_inches="tight", pad_inches=0.3)
    print(f"  Saved {filename}  ({n_rows} rows)")
    plt.close(fig)


# ── Assign experiments to groups (only those in filtered) ──

groups = {
    "Discrete BVP — scale sweep & perturbed x₀": [],
    "Broyden Banded — scale sweep": [],
    "Ext. Rosenbrock & Ext. Powell": [],
    "Broyden Tridiag": [],
}

for exp_name in filtered:
    results = filtered[exp_name]
    if "Discrete BVP" in exp_name:
        groups["Discrete BVP — scale sweep & perturbed x₀"].append((exp_name, results))
    elif "Broyden Banded" in exp_name:
        groups["Broyden Banded — scale sweep"].append((exp_name, results))
    elif "Rosenbrock" in exp_name or "Powell" in exp_name:
        groups["Ext. Rosenbrock & Ext. Powell"].append((exp_name, results))
    elif "Tridiag" in exp_name:
        groups["Broyden Tridiag"].append((exp_name, results))

filenames = {
    "Discrete BVP — scale sweep & perturbed x₀": "fig_discrete_bvp.png",
    "Broyden Banded — scale sweep": "fig_broyden_banded.png",
    "Ext. Rosenbrock & Ext. Powell": "fig_rosenbrock_powell.png",
    "Broyden Tridiag": "fig_broyden_tridiag.png",
}

print("\nGenerating figures...")
for group_name, exp_list in groups.items():
    if exp_list:
        make_group_figure(group_name, exp_list, filenames[group_name])

print("\nDone.")

  SP-Broyden Benchmark  ·  B₀ = I  ·  tol = 1e-10  ·  maxiter = 500

  Discrete BVP  n=10  (n=10)
    Broyden(p=0)            ✗ iter=18   fev=20   ‖F‖=1.79e-08
    Anderson(m=5)           ✓ iter=81   fev=82   ‖F‖=5.77e-11
    Anderson(m=10)          ✓ iter=13   fev=14   ‖F‖=5.78e-11
    SP-Broyden(p≤3)         ✓ iter=15   fev=16   ‖F‖=3.47e-11
    SP-Broyden(p≤5)         ✓ iter=14   fev=15   ‖F‖=2.77e-12
    SP-Broyden(p≤10)        ✓ iter=13   fev=14   ‖F‖=5.78e-11

  Discrete BVP  n=20  (n=20)
    Broyden(p=0)            ✗ iter=36   fev=38   ‖F‖=3.97e-09
    Anderson(m=5)           ✓ iter=329  fev=330  ‖F‖=7.62e-11
    Anderson(m=10)          ✓ iter=112  fev=113  ‖F‖=9.66e-11
    SP-Broyden(p≤3)         ✓ iter=27   fev=28   ‖F‖=3.66e-12
    SP-Broyden(p≤5)         ✓ iter=25   fev=26   ‖F‖=8.10e-13
    SP-Broyden(p≤10)        ✓ iter=23   fev=24   ‖F‖=5.01e-12

  Discrete BVP  n=50  (n=50)
    Broyden(p=0)            ✗ iter=133  fev=135  ‖F‖=1.92e-09
    Anderson(m=5)           ✗ iter=5

In [27]:
"""
SP-Broyden: данные и графики для тезисов конференции МФТИ.
Результат: fig_tezisy.png
"""

import numpy as np
from numpy.linalg import norm, solve, cond
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════
#  Единый солвер SP-Broyden
# ═══════════════════════════════════════════

def sp_broyden_solve(F, x0, p_max=0, reset=False,
                     cond_thresh=1e12, maxiter=500, tol=1e-13):
    """
    Параметры:
      p_max=0,  reset=False  →  классический Бройден
      p_max>0,  reset=False  →  SP-Broyden (секанто-сохраняющий)
      p_max=m,  reset=True   →  Андерсон (мультисекущая пересборка)
    Возвращает: list of (iteration, f_evals, ||F||)
    """
    n = len(x0)
    x = x0.copy().astype(float)
    Fx = F(x)
    f_evals = 1
    hist = [(0, 1, float(norm(Fx)))]

    B = np.eye(n)
    S_hist, Y_hist = [], []

    for k in range(maxiter):
        if norm(Fx) < tol:
            break

        try:
            d = solve(B, -Fx)
        except np.linalg.LinAlgError:
            break
        if not np.all(np.isfinite(d)):
            break

        x_new = x + d
        Fx_new = F(x_new)
        f_evals += 1
        if not np.all(np.isfinite(Fx_new)):
            break

        y = Fx_new - Fx
        s = d
        S_hist.append(s.copy())
        Y_hist.append(y.copy())

        if reset:
            # ── Ветка Андерсона: пересборка B = I + (Y-S)(S^T S)^{-1} S^T ──
            m = min(p_max, len(S_hist))
            B = np.eye(n)
            if m > 0:
                Sm = np.column_stack(S_hist[-m:])
                Ym = np.column_stack(Y_hist[-m:])
                G = Sm.T @ Sm
                while m > 0 and cond(G) > cond_thresh:
                    m -= 1
                    if m == 0:
                        break
                    Sm = np.column_stack(S_hist[-m:])
                    Ym = np.column_stack(Y_hist[-m:])
                    G = Sm.T @ Sm
                if m > 0:
                    try:
                        B = np.eye(n) + (Ym - Sm) @ solve(G, Sm.T)
                    except np.linalg.LinAlgError:
                        B = np.eye(n)
        else:
            # ── Ветка Бройдена: ранг-1 с выбором v ──
            Bs = B @ s
            p = 0
            if p_max > 0 and len(S_hist) >= 2:
                for p_try in range(1, min(p_max, len(S_hist) - 1) + 1):
                    cols = [S_hist[-1 - j] for j in range(p_try + 1)]
                    Sp = np.column_stack(cols)
                    G = Sp.T @ Sp
                    if cond(G) < cond_thresh:
                        p = p_try
                    else:
                        break

            if p == 0:
                v = s  # классический Бройден
            else:
                cols = [S_hist[-1 - j] for j in range(p + 1)]
                Sp = np.column_stack(cols)
                G = Sp.T @ Sp
                e1 = np.zeros(p + 1)
                e1[0] = 1.0
                try:
                    v = Sp @ solve(G, e1)
                except np.linalg.LinAlgError:
                    v = s

            denom = v @ s
            if abs(denom) < 1e-12:
                v = s
                denom = s @ s
            # if abs(denom) < 1e-12:
            #     break

            B = B + np.outer(y - Bs, v) / denom

        x = x_new
        Fx = Fx_new
        hist.append((k + 1, f_evals, float(norm(Fx))))

        max_hist = max(p_max + 5, 15)
        if len(S_hist) > max_hist:
            S_hist.pop(0)
            Y_hist.pop(0)

    return hist


# ═══════════════════════════════════════════
#  Тестовые задачи
# ═══════════════════════════════════════════

def discrete_bvp(x):
    n = len(x)
    h = 1.0 / (n + 1)
    r = np.zeros(n)
    for i in range(n):
        ti = (i + 1) * h
        xm = x[i - 1] if i > 0 else 0.0
        xp = x[i + 1] if i < n - 1 else 0.0
        r[i] = 2 * x[i] - xm - xp + h**2 * (x[i] + ti + 1)**3 / 2
    return r

def discrete_bvp_x0(n):
    h = 1.0 / (n + 1)
    return np.array([i * h * (i * h - 1) for i in range(1, n + 1)]) * 0.1

def broyden_banded(x):
    n = len(x)
    r = np.zeros(n)
    for i in range(n):
        ji = [j for j in range(max(0, i - 5), min(n, i + 2)) if j != i]
        r[i] = x[i] * (2 + 5 * x[i]**2) + 1 - sum(x[j] * (1 + x[j]) for j in ji)
    return r

def broyden_banded_x0(n):
    return -np.ones(n) * 0.1


# ═══════════════════════════════════════════
#  Методы для сравнения
# ═══════════════════════════════════════════

METHODS = [
    # (label, p_max, reset, color, linestyle, linewidth)
    ("Бройден",          0,  False, "#888888", (0, (4, 3)),  1.3),
    ("Андерсон (m=10)",  10, True,  "#2060B0", (0, (5, 2)),  1.5),
    ("SP-Broyden (p≤10)", 10, False, "#D03030", "-",         2.3),
]

PROBLEMS = [
    ("Discrete BVP, n = 100",   discrete_bvp,    discrete_bvp_x0(100), 1e-15),
    ("Broyden Banded, n = 100", broyden_banded,  broyden_banded_x0(100), 1e-12),
]


# ═══════════════════════════════════════════
#  Запуск и печать результатов
# ═══════════════════════════════════════════

print("=" * 70)
print("  SP-Broyden: данные для тезисов  |  B_0 = I  |  tol = 1e-10")
print("=" * 70)

all_data = {}

for prob_name, F, x0, tol in PROBLEMS:
    print(f"\n  {prob_name}")
    prob_data = {}
    for label, pm, rst, *_ in METHODS:
        h = sp_broyden_solve(F, x0.copy(), p_max=pm, reset=rst, tol=tol)
        final_res = h[-1][2]
        n_iter = h[-1][0]
        conv = final_res < 1e-10
        print(f"    {label:<22s}  iter={n_iter:<4d}  ||F||={final_res:.2e}")
        prob_data[label] = h
    all_data[prob_name] = prob_data


# ═══════════════════════════════════════════
#  Построение графиков
# ═══════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.2, 2.4))

for ax, (prob_name, F, x0, tol) in zip([ax1, ax2], PROBLEMS):
    data = all_data[prob_name]

    for label, pm, rst, color, ls, lw in METHODS:
        h = data[label]
        iters = [t[0] for t in h]
        resid = [t[2] for t in h]

        # Фильтруем inf/nan
        xs, ys = [], []
        for i, r in zip(iters, resid):
            if np.isfinite(r) and r > 0:
                xs.append(i)
                ys.append(r)
            else:
                break

        if len(xs) < 2:
            continue

        # Метка с числом итераций при сходимости
        tag = ""
        if ys[-1] < 1e-10:
            tag = f" ({xs[-1]})"

        ax.semilogy(xs, ys, color=color, linestyle=ls,
                    linewidth=lw, label=f"{label}", alpha=0.9)

    # Линия допуска
    ax.axhline(1e-10, color="#00BBCC", lw=0.5, ls=":", alpha=0.4)

    ax.set_title(prob_name, fontsize=8, pad=4)
    ax.set_xlabel("итерация k", fontsize=7)
    ax.set_ylabel("‖F(x)‖", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)

# ── Обрезка сверху: показываем только зону сходимости ──
ax2.set_ylim(bottom=1e-14, top=1e8)

ax2.legend(fontsize=5.5, loc="upper right", framealpha=0.8,
           borderpad=0.3, handlelength=1.5, labelspacing=0.2)

fig.tight_layout(pad=0.5)
fig.savefig("fig_tezisy.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
print(f"\nГрафик сохранен: fig_tezisy.png")
plt.close()

  SP-Broyden: данные для тезисов  |  B_0 = I  |  tol = 1e-10

  Discrete BVP, n = 100
    Бройден                 iter=371   ||F||=9.85e-16
    Андерсон (m=10)         iter=500   ||F||=2.18e-04
    SP-Broyden (p≤10)       iter=115   ||F||=2.42e-16

  Broyden Banded, n = 100
    Бройден                 iter=48    ||F||=inf
    Андерсон (m=10)         iter=242   ||F||=6.96e-13
    SP-Broyden (p≤10)       iter=144   ||F||=8.72e-13

График сохранен: fig_tezisy.png
